# DengueGuard BD
### ML-based Dengue Risk Prediction & Analysis for Bangladesh

এই নোটবুকে আমরা বাংলাদেশের একটি real dengue dataset ব্যবহার করে ডেঙ্গু predict করার জন্য
৩টা মডেল বানাব, তুলনা করব, আর দেখব কোনটা সবচেয়ে ভালো।

**Dataset:** *A Comprehensive Dengue Dataset of Bangladesh*, Mendeley Data (2025), Md Kawsar Ahmad
[data.mendeley.com/datasets/zdtc3n6xv2](https://data.mendeley.com/datasets/zdtc3n6xv2)

**ধাপ:**
1. Setup & Data Load
2. Data Cleaning
3. Exploratory Data Analysis (EDA)
4. Preprocessing
5. Model Training (Logistic Regression, Random Forest, XGBoost)
6. Model Evaluation & Comparison
7. Feature Importance
8. Save the Best Model


## 1. Setup

**Colab-এ প্রথমে এই সেল চালান।** যদি Colab এ চালান, তাহলে `dataset.csv` ফাইলটা আগে আপলোড করে নিন
(বাম দিকে ফোল্ডার আইকনে ক্লিক করে, অথবা নিচের সেল দিয়ে)।


In [ ]:
# যদি Google Colab-এ চালান, dataset.csv আপলোড করতে এই সেলটা চালান
# (আপনার কম্পিউটার থেকে data/dataset.csv ফাইলটা বেছে নিন)
try:
    from google.colab import files
    import os
    if not os.path.exists("dataset.csv"):
        uploaded = files.upload()
except ImportError:
    print("Google Colab না হলে এই সেল স্কিপ করুন, dataset.csv আগে থেকেই একই ফোল্ডারে আছে ধরে নিচ্ছি।")


In [ ]:
# প্রয়োজনীয় লাইব্রেরি ইনস্টল (Colab-এ সাধারণত সবগুলো আগে থেকেই থাকে)
!pip install -q xgboost shap


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, roc_curve, auc,
                              classification_report, ConfusionMatrixDisplay)
import joblib

sns.set_style("whitegrid")
RANDOM_STATE = 42


## 2. Data Load & Cleaning

In [ ]:
import os
csv_path = "dataset.csv" if os.path.exists("dataset.csv") else "data/dataset.csv"
df = pd.read_csv(csv_path)
print(df.shape)
df.head()


In [ ]:
df.info()
df.isnull().sum()


**Cleaning সিদ্ধান্ত:**
- `District` কলামে সব মান একই (শুধু "Dhaka"), তাই কোনো predictive signal নেই -> drop করব
- `Joint_Pain` কলামে missing value আছে, এর মানে ধরে নিচ্ছি রোগীর joint pain ছিল না -> `"None"` দিয়ে fill করব


In [ ]:
df = df.drop(columns=["District"])
df["Joint_Pain"] = df["Joint_Pain"].fillna("None")
df.isnull().sum().sum()  # 0 হওয়া উচিত


## 3. Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(5,4))
sns.countplot(data=df, x="Outcome", hue="Outcome", palette=["#5cb85c","#d9534f"], legend=False)
plt.xticks([0,1], ["No Dengue", "Dengue"])
plt.title("Outcome Distribution")
plt.show()
df["Outcome"].value_counts(normalize=True)


In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(data=df, x="Age", hue="Outcome", multiple="stack", bins=20, palette=["#5cb85c","#d9534f"])
plt.title("Age Distribution by Outcome")
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
area_outcome = df.groupby("Area")["Outcome"].mean().sort_values(ascending=False)
sns.barplot(x=area_outcome.values, y=area_outcome.index, color="#d9534f")
plt.title("Dengue Positive Rate by Area (Dhaka)")
plt.xlabel("Positive rate")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10,4))
sns.countplot(data=df, x="AreaType", hue="Outcome", palette=["#5cb85c","#d9534f"], ax=axes[0])
axes[0].set_title("Outcome by Area Type")
sns.countplot(data=df, x="HouseType", hue="Outcome", palette=["#5cb85c","#d9534f"], ax=axes[1])
axes[1].set_title("Outcome by House Type")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(data=df, x="Outcome", y="Body_Temperature", hue="Outcome", palette=["#5cb85c","#d9534f"], legend=False)
plt.xticks([0,1], ["No Dengue", "Dengue"])
plt.title("Body Temperature vs Outcome")
plt.show()


**পর্যবেক্ষণ:** Body Temperature, Platelet Count আর Fever Duration-এর মধ্যে dengue পজিটিভ ও নেগেটিভ
গ্রুপের মধ্যে স্পষ্ট পার্থক্য দেখা যাচ্ছে (প্রায় কোনো overlap নেই)। এটা এই public dataset-টির একটা
বৈশিষ্ট্য (পরিষ্কার/synthetic ধরনের ডেটা), বাস্তব হাসপাতালের ডেটায় সাধারণত এতটা পরিষ্কার বিভাজন থাকে না।
এই কারণেই আমরা দুই ধরনের মডেল বানাব।


## 4. Two Modeling Scenarios

- **Screening Model**: শুধু demographics, location, আর symptoms দিয়ে (কোনো lab test ছাড়া)।
  বাস্তব ব্যবহার: রোগী হাসপাতালে আসার সাথে সাথে, টেস্ট রিপোর্ট আসার আগেই ঝুঁকি অনুমান করা।
- **Confirmatory Model**: NS1/IgG/IgM ল্যাব টেস্ট রেজাল্টসহ। এটা ডাক্তারের চূড়ান্ত সিদ্ধান্তের কাছাকাছি।

আমরা মূলত **Screening Model**-টাই আমাদের Streamlit অ্যাপে ব্যবহার করব, কারণ বাস্তব জীবনে
এটাই বেশি কাজে লাগে (টেস্ট করার আগেই একটা ধারণা পাওয়া)।


In [ ]:
TARGET = "Outcome"
NUMERIC_FEATURES = ["Age", "Fever_Duration", "Body_Temperature", "Platelet_Count", "WBC_Count"]
SYMPTOM_FEATURES = ["Headache", "Retro_Orbital_Pain", "Myalgia", "Rash"]
CATEGORICAL_FEATURES = ["Gender", "Area", "AreaType", "HouseType", "Joint_Pain"]
LAB_FEATURES = ["NS1", "IgG", "IgM"]

SCREENING_FEATURES = NUMERIC_FEATURES + SYMPTOM_FEATURES + CATEGORICAL_FEATURES
FULL_FEATURES = SCREENING_FEATURES + LAB_FEATURES


In [ ]:
def build_models():
    return {
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        "Random Forest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
        "XGBoost": XGBClassifier(n_estimators=300, eval_metric="logloss", random_state=RANDOM_STATE),
    }

def run_experiment(features, tag):
    X = df[features]
    y = df[TARGET]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )
    preprocessor = ColumnTransformer(
        transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES)],
        remainder="passthrough"
    )

    results = {}
    fitted = {}

    for name, clf in build_models().items():
        pipe = Pipeline([("prep", preprocessor), ("clf", clf)])
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)
        y_proba = pipe.predict_proba(X_test)[:, 1]

        results[name] = {
            "accuracy": round(accuracy_score(y_test, y_pred), 4),
            "precision": round(precision_score(y_test, y_pred), 4),
            "recall": round(recall_score(y_test, y_pred), 4),
            "f1": round(f1_score(y_test, y_pred), 4),
        }
        fitted[name] = pipe

        print(f"--- {tag} | {name} ---")
        print(classification_report(y_test, y_pred, target_names=["No Dengue", "Dengue"]))

        ConfusionMatrixDisplay.from_predictions(
            y_test, y_pred, display_labels=["No Dengue", "Dengue"], cmap="Blues"
        )
        plt.title(f"Confusion Matrix - {name} ({tag})")
        plt.show()

    return results, fitted, X_test, y_test


In [ ]:
screening_results, screening_fitted, X_test_s, y_test_s = run_experiment(SCREENING_FEATURES, "screening")
pd.DataFrame(screening_results).T


In [ ]:
full_results, full_fitted, X_test_f, y_test_f = run_experiment(FULL_FEATURES, "full")
pd.DataFrame(full_results).T


## 5. Model Comparison (ROC Curve)

In [ ]:
plt.figure(figsize=(6,5))
for name, pipe in screening_fitted.items():
    y_proba = pipe.predict_proba(X_test_s)[:, 1]
    fpr, tpr, _ = roc_curve(y_test_s, y_proba)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc:.2f})")
plt.plot([0,1],[0,1],"k--", alpha=0.4)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Screening Models")
plt.legend()
plt.show()


## 6. Feature Importance (best screening model)

In [ ]:
best_screening_name = max(screening_results, key=lambda k: screening_results[k]["f1"])
print("Best screening model:", best_screening_name)

best_pipe = screening_fitted[best_screening_name]
try:
    clf = best_pipe.named_steps["clf"]
    prep = best_pipe.named_steps["prep"]
    feat_names = prep.get_feature_names_out()
    importances = clf.feature_importances_
    imp_df = pd.DataFrame({"feature": feat_names, "importance": importances}).sort_values(
        "importance", ascending=False).head(12)
    plt.figure(figsize=(7,5))
    sns.barplot(data=imp_df, x="importance", y="feature", color="#d9534f")
    plt.title(f"Top Feature Importances - {best_screening_name}")
    plt.tight_layout()
    plt.show()
except AttributeError:
    print(f"{best_screening_name} doesn't expose feature_importances_ (e.g. Logistic Regression). "
          "Try Random Forest or XGBoost instead to see this plot.")


## 7. SHAP Explainability (optional, best tree model)

SHAP দেখায় কোন feature কতটা prediction-কে push করছে (positive বা negative দিকে)।
এটা ইন্টারভিউতে "black box model কেন বিশ্বাস করব?" প্রশ্নের চমৎকার উত্তর।


In [ ]:
import shap

tree_name = "XGBoost" if "XGBoost" in screening_fitted else "Random Forest"
tree_pipe = screening_fitted[tree_name]
prep = tree_pipe.named_steps["prep"]
clf = tree_pipe.named_steps["clf"]

X_test_transformed = prep.transform(X_test_s)
feat_names = prep.get_feature_names_out()

explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test_transformed)

shap.summary_plot(shap_values, X_test_transformed, feature_names=feat_names, show=True)


## 8. Save the Best Model (for the Streamlit app)

In [ ]:
joblib.dump(best_pipe, "dengueguard_model.joblib")
joblib.dump(SCREENING_FEATURES, "feature_list.joblib")
joblib.dump(best_screening_name, "best_model_name.joblib")

import json
with open("results.json", "w") as f:
    json.dump({
        "screening": {"results": screening_results, "best_model": best_screening_name, "features": SCREENING_FEATURES},
        "full": {"results": full_results, "best_model": max(full_results, key=lambda k: full_results[k]["f1"]), "features": FULL_FEATURES},
    }, f, indent=2)

print("Saved: dengueguard_model.joblib, feature_list.joblib, best_model_name.joblib, results.json")
print("এই ফাইলগুলো model/ ফোল্ডারে কপি করে GitHub repo-তে রাখুন, Streamlit app.py এগুলোই ব্যবহার করে।")


## 9. Summary / Conclusion

- ডেটাসেট: ১,০০০ রোগীর রেকর্ড, বাংলাদেশ (ঢাকা), demographics + clinical + serological তথ্য
- Screening model (কোনো ল্যাব টেস্ট ছাড়া) দিয়েও ভালো ফলাফল পাওয়া গেছে, কারণ vitals (temperature, platelet)
  খুব strong indicator
- Confirmatory model (ল্যাব টেস্ট সহ) আরও নির্ভরযোগ্য, বাস্তব ক্লিনিক্যাল সিদ্ধান্তের জন্য উপযুক্ত
- সেরা মডেল সেভ করে Streamlit ওয়েব অ্যাপে ব্যবহার করা হয়েছে

**পরবর্তী উন্নতির সুযোগ:** আরও জেলার ডেটা যোগ করা, cross-validation, hyperparameter tuning,
এবং external validation dataset দিয়ে টেস্ট করা।
